# 🧵 Demo efímera — Tejido Empresarial · ProColombia · Colab + Cloudflare

Aplicativo de **Tejido Empresarial** (GIC · ProColombia) para los ejes de
**Exportaciones, Inversión y Turismo**: segmentación con filtros dependientes,
búsqueda por razón social / NIT / lote, ficha de empresa, glosario y descarga
Excel con formato.

A diferencia del sitio de la Célula de IA (estático), este aplicativo tiene
**dos piezas**: el frontend React que se compila a archivos estáticos y la
**API FastAPI** que consulta Snowflake y genera los Excel. Este notebook levanta
las dos dentro de Colab y **TryCloudflare** entrega una URL pública temporal
(sin cuenta ni token), igual que en las demos de Potencial.

| `MODO_DATOS` | Qué muestra | Requiere | Cuándo usarlo |
|---|---|---|---|
| `"demo"` (por defecto) | 14 empresas **sintéticas** | nada | Revisar diseño, navegación, formato del Excel |
| `"snowflake"` | Datos **reales** de la base | secretos `SF_*` en Colab | Validación final antes de Railway |

**Tiempo**: 3–5 min la primera vez (instala Node y compila el frontend);
~40 s en las siguientes ejecuciones de la misma sesión.

> ⚠️ **Límites no negociables**
> 1. La [FAQ de Colab](https://research.google.com/colaboratory/faq.html) prohíbe
>    usar los runtimes como hosting: esto es una demo corta con usted presente.
>    Al terminar ejecute `detener_todo()`.
> 2. La URL `*.trycloudflare.com` es efímera: cambia en cada sesión y muere con
>    el runtime. El enlace estable es **Railway** (última celda).
> 3. Con `MODO_DATOS="snowflake"` la URL pública expone **datos empresariales
>    reales, incluidos contactos**. Por eso el notebook activa por defecto una
>    contraseña de acceso y la imprime aquí. No comparta la URL sin ella.


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PASO 0 — DEFINICIONES · ejecutar una vez por sesión · no editar
# ══════════════════════════════════════════════════════════════════
"""Demo efímera del aplicativo Tejido Empresarial (React + FastAPI).

Garantías de diseño:
- Localiza el proyecto por marcadores únicos (backend/config.py +
  frontend/package.json + Dockerfile): imposible servir otro paquete.
- Copia SIEMPRE a disco local (/content): nunca compila ni sirve sobre Drive.
- Verifica /api/health y que la portada contenga «Tejido Empresarial» antes
  de entregar la URL.
- Con datos reales protege la URL con usuario y contraseña generados aquí.
- Todo queda en logs (/content/*.log); diagnostico() los muestra.
"""
from __future__ import annotations

import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from base64 import b64encode
from pathlib import Path

# ── Constantes ──────────────────────────────────────────────────────
MARCADORES = ["backend/config.py", "frontend/package.json", "Dockerfile"]
EXCLUIR_COPIA = {"node_modules", ".git", ".ipynb_checkpoints", "__pycache__",
                 ".pytest_cache", ".venv", "venv", ".vite", "z. Mejoras",
                 "legado_streamlit", "setup", "previews"}
DIR_APP    = Path("/content/tejido-app")
LOG_NODE   = "/content/build_frontend.log"
LOG_PIP    = "/content/pip.log"
LOG_API    = "/content/api.log"
LOG_TUNEL  = "/content/tunel.log"
CLOUDFLARED = "/content/cloudflared"
SECRETOS_SF = ["SF_ACCOUNT", "SF_USER", "SF_DATABASE", "SF_SCHEMA",
               "SF_WAREHOUSE", "SF_ROLE", "SF_PRIVATE_KEY_B64_1"]
SECRETOS_SF_OPCIONALES = ["SF_PRIVATE_KEY_PASSPHRASE_1",
                          "SF_PRIVATE_KEY_B64_2", "SF_PRIVATE_KEY_PASSPHRASE_2"]

_PROCESOS: dict[str, subprocess.Popen] = {}
_ACCESO: dict[str, str] = {}      # usuario/clave si la demo queda protegida


def _sh(cmd: list[str] | str, log: str | None = None, cwd: str | Path | None = None) -> None:
    """Ejecuta un comando; si falla, muestra el final del log y aborta."""
    shell = isinstance(cmd, str)
    with (open(log, "a") if log else open(os.devnull, "w")) as f:
        r = subprocess.run(cmd, shell=shell, stdout=f, stderr=subprocess.STDOUT,
                           cwd=str(cwd) if cwd else None)
    if r.returncode != 0:
        if log and Path(log).exists():
            print("\n".join(Path(log).read_text().splitlines()[-25:]))
        mostrado = cmd if shell else " ".join(cmd)
        raise SystemExit(f"✗ Falló: {mostrado} (código {r.returncode})")


def montar_drive() -> None:
    if Path("/content/drive/MyDrive").exists():
        print("✓ Drive ya estaba montado")
        return
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    print("✓ Drive montado")


def localizar_proyecto(ruta_drive: str) -> Path:
    """Busca (hasta 3 niveles) la carpeta con TODOS los marcadores."""
    base = Path(ruta_drive)
    if not base.exists():
        raise SystemExit(f"✗ No existe la ruta en su Drive: {ruta_drive}")

    def es_raiz(p: Path) -> bool:
        return all((p / m).exists() for m in MARCADORES)

    if es_raiz(base):
        print(f"✓ Proyecto localizado: {base}")
        return base
    nivel = [base]
    for _ in range(3):
        siguiente = []
        for c in nivel:
            if not c.is_dir():
                continue
            for h in sorted(c.iterdir()):
                if not h.is_dir() or h.name in EXCLUIR_COPIA:
                    continue
                if es_raiz(h):
                    print(f"✓ Proyecto localizado: {h}")
                    return h
                siguiente.append(h)
        nivel = siguiente
    raise SystemExit(
        f"✗ Ninguna carpeta bajo {ruta_drive} contiene {MARCADORES}.\n"
        "  ¿Descomprimió el paquete completo en esa ruta?"
    )


def copiar_local(origen: Path) -> Path:
    """Copia el proyecto de Drive a disco local (sin node_modules ni legado)."""
    if DIR_APP.exists():
        shutil.rmtree(DIR_APP)
    print(f"→ Copiando {origen} → {DIR_APP} …")
    shutil.copytree(origen, DIR_APP, ignore=shutil.ignore_patterns(*EXCLUIR_COPIA))
    n = sum(1 for _ in DIR_APP.rglob("*") if _.is_file())
    print(f"✓ Copia local lista ({n} archivos)")
    return DIR_APP


def asegurar_node(mayor: int = 22) -> None:
    try:
        v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
        if v and int(v.lstrip("v").split(".")[0]) >= mayor:
            print(f"✓ Node {v}")
            return
    except FileNotFoundError:
        v = "ausente"
    print(f"→ Node {v}: instalando Node {mayor} (1–2 min) …")
    _sh(f"curl -fsSL https://deb.nodesource.com/setup_{mayor}.x | bash - >>{LOG_NODE} 2>&1", LOG_NODE)
    _sh(["apt-get", "install", "-y", "nodejs"], LOG_NODE)
    print("✓ Node", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip())


def compilar_frontend(app: Path, forzar: bool = False) -> Path:
    """npm ci + npm run build en frontend/. Devuelve la ruta de dist/."""
    front = app / "frontend"
    dist = front / "dist"
    if dist.joinpath("index.html").exists() and not forzar:
        print("✓ frontend/dist ya existe (use FORZAR_RECOMPILAR=True para rehacerlo)")
        return dist
    asegurar_node(22)
    Path(LOG_NODE).write_text("")
    instalar = "npm ci --no-audit --no-fund" if (front / "package-lock.json").exists() else "npm install --no-audit --no-fund"
    print(f"→ {instalar} … (1–2 min)")
    _sh(f"{instalar} >>{LOG_NODE} 2>&1", LOG_NODE, cwd=front)
    print("→ npm run build … (tipos + producción)")
    _sh(f"npm run build >>{LOG_NODE} 2>&1", LOG_NODE, cwd=front)
    html = dist / "index.html"
    if not html.exists() or "Tejido Empresarial" not in html.read_text(encoding="utf-8"):
        raise SystemExit(f"✗ La compilación no produjo un dist/ válido — vea diagnostico()")
    print(f"✓ Frontend compilado: {dist}")
    return dist


def instalar_dependencias(modo_datos: str) -> None:
    """Instala lo mínimo para correr la API en Colab.

    Railway usa requirements-api.txt con versiones fijas; aquí se instala el
    conjunto equivalente sin fijar versiones para no pelear con las que Colab
    ya trae (pandas, por ejemplo).
    """
    Path(LOG_PIP).write_text("")
    paquetes = ["fastapi", "uvicorn[standard]", "python-dotenv", "openpyxl", "XlsxWriter"]
    try:
        import pandas  # noqa: F401
    except ImportError:
        paquetes.append("pandas")
    print("→ pip install", " ".join(paquetes), "…")
    _sh(f'{sys.executable} -m pip install -q {" ".join(chr(34) + p + chr(34) for p in paquetes)} >>{LOG_PIP} 2>&1', LOG_PIP)
    if modo_datos == "snowflake":
        print("→ pip install snowflake-snowpark-python … (1–3 min)")
        try:
            _sh(f"{sys.executable} -m pip install -q snowflake-snowpark-python >>{LOG_PIP} 2>&1", LOG_PIP)
        except SystemExit:
            raise SystemExit(
                "✗ No se pudo instalar el conector de Snowflake en este runtime.\n"
                f"  Python de Colab: {sys.version.split()[0]}. El conector aún no soporta\n"
                "  todas las versiones nuevas. Opciones:\n"
                "   a) Entorno de ejecución → Cambiar tipo → (elija una versión de Python compatible), o\n"
                '   b) use MODO_DATOS="demo" para revisar la interfaz y valide los datos reales en Railway.'
            )
    print("✓ Dependencias listas")


def _secreto(nombre: str, obligatorio: bool = True) -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            v = userdata.get(nombre)
        except Exception:
            v = None
    except Exception:
        v = os.environ.get(nombre)
    v = (v or "").strip()
    if not v and obligatorio:
        raise SystemExit(
            f"✗ Falta el secreto '{nombre}'. Panel izquierdo → 🔑 Secretos → añádalo\n"
            "  y active «Acceso del cuaderno». Son los mismos valores que usa Railway."
        )
    return v


def preparar_entorno(modo_datos: str, proteger: bool, incluir_contactos: bool) -> dict[str, str]:
    """Variables de entorno con las que se ejecutará la API."""
    env = os.environ.copy()
    env["APP_ENV"] = "development"          # habilita /api/docs para la revisión
    env["EXPORT_INCLUDE_CONTACT_FIELDS"] = "true" if incluir_contactos else "false"
    env.pop("PUBLIC_ORIGIN", None)

    if modo_datos == "demo":
        env["APP_DEMO_MODE"] = "true"
        print("✓ Modo DEMOSTRACIÓN: 14 empresas sintéticas, sin Snowflake")
    else:
        env["APP_DEMO_MODE"] = "false"
        for nombre in SECRETOS_SF:
            env[nombre] = _secreto(nombre)
        for nombre in SECRETOS_SF_OPCIONALES:
            valor = _secreto(nombre, obligatorio=False)
            if valor:
                env[nombre] = valor
        print(f"✓ Modo SNOWFLAKE: {env['SF_DATABASE']}.{env['SF_SCHEMA']} · rol {env['SF_ROLE']}")

    if proteger:
        _ACCESO["usuario"] = "procolombia"
        _ACCESO["clave"] = secrets.token_urlsafe(9)
        env["APP_BASIC_USER"] = _ACCESO["usuario"]
        env["APP_BASIC_PASSWORD"] = _ACCESO["clave"]
        print("✓ La URL quedará protegida con usuario y contraseña (se imprimen en el Paso 2)")
    else:
        env.pop("APP_BASIC_USER", None)
        env.pop("APP_BASIC_PASSWORD", None)
        _ACCESO.clear()
    return env


def _cabeceras() -> dict[str, str]:
    if not _ACCESO:
        return {}
    par = f"{_ACCESO['usuario']}:{_ACCESO['clave']}".encode()
    return {"Authorization": "Basic " + b64encode(par).decode()}


def _pedir(url: str, timeout: int = 10) -> tuple[int, str]:
    pedido = urllib.request.Request(url, headers=_cabeceras())
    try:
        with urllib.request.urlopen(pedido, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", "ignore")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", "ignore")


def lanzar_api(app: Path, env: dict[str, str], puerto: int = 8000) -> None:
    detener_todo(silencioso=True)
    Path(LOG_API).write_text("")
    _PROCESOS["api"] = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend.main:app",
         "--host", "127.0.0.1", "--port", str(puerto), "--log-level", "warning"],
        cwd=str(app), env=env,
        stdout=open(LOG_API, "a"), stderr=subprocess.STDOUT,
    )
    salud = None
    for _ in range(60):
        if _PROCESOS["api"].poll() is not None:
            print("\n".join(Path(LOG_API).read_text().splitlines()[-25:]))
            raise SystemExit("✗ La API se detuvo al arrancar — vea el log de arriba")
        try:
            codigo, cuerpo = _pedir(f"http://127.0.0.1:{puerto}/api/health", timeout=3)
            if codigo == 200:
                salud = json.loads(cuerpo)
                break
        except Exception:
            time.sleep(0.5)
    if not salud:
        raise SystemExit("✗ La API no respondió en 30 s — vea diagnostico()")
    print(f"✓ API viva · versión {salud['version']} · datos: {salud['data_connection']} · acceso: {salud['access_control']}")
    if not salud.get("frontend_built"):
        raise SystemExit("✗ La API no encuentra frontend/dist — vuelva a ejecutar el Paso 1")

    codigo, html = _pedir(f"http://127.0.0.1:{puerto}/")
    if codigo != 200 or "Tejido Empresarial" not in html:
        raise SystemExit(f"✗ La portada no respondió como se esperaba (HTTP {codigo})")
    print("✓ Portada verificada")


def verificar_snowflake(puerto: int = 8000) -> None:
    """Prueba real contra Snowflake (equivale a /api/health?deep=true)."""
    codigo, cuerpo = _pedir(f"http://127.0.0.1:{puerto}/api/health?deep=true", timeout=90)
    if codigo == 200 and json.loads(cuerpo).get("data_connection") == "connected":
        print("✓ Snowflake respondió: la conexión funciona de extremo a extremo")
    else:
        print(f"✗ Snowflake no respondió (HTTP {codigo}): {cuerpo[:300]}")
        print("  Revise los secretos SF_* y que la llave sea la vigente.")


def probar_api(puerto: int = 8000) -> None:
    """Prueba de humo por HTTP: metadatos, búsqueda, ficha y descarga."""
    base = f"http://127.0.0.1:{puerto}"
    print("─" * 62)
    codigo, cuerpo = _pedir(f"{base}/api/metadata")
    meta = json.loads(cuerpo)
    print(f"  Metadatos     : {len(meta['filters'])} filtros · {len(meta['export_columns'])} variables · corte {meta['periods']['companies']}")

    pedido = urllib.request.Request(
        f"{base}/api/companies/search", method="POST",
        data=json.dumps({"mode": "filters", "filters": {}, "page": 1, "page_size": 25}).encode(),
        headers={"Content-Type": "application/json", **_cabeceras()},
    )
    with urllib.request.urlopen(pedido, timeout=120) as r:
        datos = json.loads(r.read().decode())
    print(f"  Búsqueda      : {datos['total']:,} empresas · {len(datos['rows'])} en la primera página".replace(",", "."))

    if datos["rows"]:
        nit = str(datos["rows"][0]["NIT"])
        codigo, _ = _pedir(f"{base}/api/companies/{nit}", timeout=60)
        print(f"  Ficha {nit:<10}: HTTP {codigo}")
        pedido = urllib.request.Request(
            f"{base}/api/companies/export", method="POST",
            data=json.dumps({"mode": "nit", "term": nit}).encode(),
            headers={"Content-Type": "application/json", **_cabeceras()},
        )
        with urllib.request.urlopen(pedido, timeout=180) as r:
            contenido = r.read()
            nombre = urllib.parse.unquote(r.headers.get("X-Export-Filename", ""))
        print(f"  Excel         : {len(contenido)/1000:.0f} kB · {nombre}")
    codigo, cuerpo = _pedir(f"{base}/api/glossary")
    glosario = json.loads(cuerpo)
    print(f"  Glosario      : {glosario['count']} variables ({glosario['institutional_count']} institucionales)")
    print("─" * 62)


def _tunel(puerto: int) -> str:
    if not Path(CLOUDFLARED).exists():
        print("→ Descargando cloudflared …")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CLOUDFLARED,
        )
        os.chmod(CLOUDFLARED, 0o755)
    Path(LOG_TUNEL).write_text("")
    _PROCESOS["tunel"] = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{puerto}", "--no-autoupdate"],
        stdout=open(LOG_TUNEL, "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path(LOG_TUNEL).read_text())
        if m:
            return m.group(0)
        time.sleep(0.5)
    raise SystemExit("✗ El túnel no entregó URL — vea diagnostico()")


def desplegar(app: Path, env: dict[str, str], puerto: int = 8000) -> str:
    lanzar_api(app, env, puerto)
    url = _tunel(puerto)
    print("\n" + "═" * 62)
    print("  URL PÚBLICA (temporal, muere con esta sesión):")
    print(f"  {url}")
    if _ACCESO:
        print()
        print(f"  Usuario   : {_ACCESO['usuario']}")
        print(f"  Contraseña: {_ACCESO['clave']}")
        print("  (el navegador la pedirá al abrir el enlace)")
    print("═" * 62)
    print("Lista de verificación (2 min):")
    print(" 1. Portada: animación del tejido y los tres ejes (Exportaciones, Inversión, Turismo)")
    print(" 2. Consultar → Segmentar con filtros → elija un Departamento y vea cómo")
    print("    se reducen los Municipios → Buscar empresas")
    print(" 3. En la tabla: ordene por una columna, cambie las columnas visibles y abra una ficha")
    print(" 4. Descargue el Excel y ábralo: Resumen · Vista_Principal · Datos_Completos · Diccionario")
    print(" 5. Glosario y Metodología cargan con sus descargas")
    print(" 6. Abra la MISMA URL en el celular: menú, cajón de filtros y tarjetas")
    print("\nAtajos: probar_api() · verificar_snowflake() · detener_todo() · diagnostico()")
    return url


def detener_todo(silencioso: bool = False) -> None:
    for nombre, p in list(_PROCESOS.items()):
        if p.poll() is None:
            p.terminate()
        _PROCESOS.pop(nombre, None)
    for patron in ("uvicorn", "cloudflared"):
        try:
            subprocess.run(["pkill", "-f", patron], capture_output=True)
        except OSError:
            pass       # fuera de Linux no existe pkill; los procesos ya se terminaron arriba
    if not silencioso:
        print("✓ API y túnel detenidos (la URL pública dejó de existir)")


def diagnostico() -> None:
    for etiqueta, ruta in [("FRONTEND (npm)", LOG_NODE), ("PIP", LOG_PIP),
                           ("API (uvicorn)", LOG_API), ("TÚNEL", LOG_TUNEL)]:
        print(f"\n──── {etiqueta} ({ruta}) " + "─" * 24)
        p = Path(ruta)
        contenido = p.read_text()[-2500:] if p.exists() else ""
        print(contenido or "(vacío)")

print("✓ Definiciones cargadas. Siga con el Paso 1.")


## Paso 1 · Configurar y preparar

`RUTA_DRIVE` = carpeta de su Drive donde descomprimió el paquete del aplicativo
(o cualquier carpeta que la contenga: el localizador busca hacia adentro hasta
3 niveles usando los marcadores del proyecto).

`MODO_DATOS`:
- `"demo"` → 14 empresas sintéticas. No toca Snowflake ni necesita secretos.
- `"snowflake"` → datos reales. Requiere estos secretos en Colab (🔑 panel
  izquierdo), con los mismos valores que configurará en Railway:
  `SF_ACCOUNT`, `SF_USER`, `SF_DATABASE`, `SF_SCHEMA`, `SF_WAREHOUSE`,
  `SF_ROLE`, `SF_PRIVATE_KEY_B64_1` (y `SF_PRIVATE_KEY_PASSPHRASE_1` si la
  llave está cifrada).

`PROTEGER_CON_CLAVE = "auto"` activa usuario y contraseña cuando los datos son
reales, y deja el acceso abierto en modo demostración. La contraseña se genera
en cada ejecución y se imprime en el Paso 2.

Esta celda compila el frontend (`npm ci` + `npm run build`, los mismos comandos
que ejecuta Railway) e instala las dependencias de la API.


In [ ]:
# ── Paso 1 · Configuración y preparación ─────────────────────────────
RUTA_DRIVE = "/content/drive/MyDrive/ProColombia/tejido_empresarial_react"  # ← su Drive
MODO_DATOS = "demo"              # "demo" | "snowflake"
FORZAR_RECOMPILAR = False        # True: rehacer el build aunque ya exista dist/
PROTEGER_CON_CLAVE = "auto"      # "auto" | True | False
INCLUIR_CONTACTOS = True         # como en producción; False oculta correo/teléfono/representante

assert MODO_DATOS in ("demo", "snowflake"), 'MODO_DATOS debe ser "demo" o "snowflake"'
_proteger = (MODO_DATOS == "snowflake") if PROTEGER_CON_CLAVE == "auto" else bool(PROTEGER_CON_CLAVE)

montar_drive()
ORIGEN = localizar_proyecto(RUTA_DRIVE)
APP = copiar_local(ORIGEN)
compilar_frontend(APP, forzar=FORZAR_RECOMPILAR)
instalar_dependencias(MODO_DATOS)
ENTORNO = preparar_entorno(MODO_DATOS, _proteger, INCLUIR_CONTACTOS)
print("\n✓ Todo listo. Siga con el Paso 2.")


## Paso 2 · Levantar el aplicativo y abrir el túnel

Arranca **uvicorn** (API + frontend compilado desde el mismo origen, igual que
en Railway), comprueba `/api/health` y la portada, descarga `cloudflared` y
abre el túnel. Al final imprime la **URL pública**, las credenciales si aplica
y la lista de verificación.


In [ ]:
# ── Paso 2 · Servir + túnel ──────────────────────────────────────────
PUERTO = 8000
URL_PUBLICA = desplegar(APP, ENTORNO, PUERTO)


## Comprobaciones rápidas (opcionales)

- `probar_api()` — prueba de humo por HTTP: metadatos, una búsqueda, una ficha,
  una descarga de Excel y el glosario. Útil para verificar sin hacer clic.
- `verificar_snowflake()` — sólo con `MODO_DATOS="snowflake"`: ejecuta la prueba
  profunda (`/api/health?deep=true`) contra la base real.


In [ ]:
# Prueba de humo por HTTP (no reemplaza la revisión visual)
probar_api()


In [ ]:
# Solo si MODO_DATOS = "snowflake": prueba real contra la base
verificar_snowflake()


## Detener · Diagnóstico

- **Terminó la demo:** ejecute `detener_todo()`. No deje el túnel abierto.
- **Algo falló:** `diagnostico()` muestra los cuatro logs (npm, pip, API, túnel).
- **Cambió el código en Drive:** vuelva a ejecutar el Paso 1 con
  `FORZAR_RECOMPILAR = True` y luego el Paso 2.


In [ ]:
# Detener la API y el túnel (cierra el enlace público)
detener_todo()


In [ ]:
# Logs de npm, pip, API y túnel (si algo no arrancó)
diagnostico()


---
# 🚂 Del Colab efímero a Railway (enlace estable)

Colab no es hosting. Cuando la revisión esté conforme, el camino es:

```
Drive → Publicacion_GitHub_TejidoEmpresarial.ipynb → GitHub → Railway
                                                       └→ redespliegue automático en cada push
```

1. Publique con el notebook hermano
   (`notebooks/Publicacion_GitHub_TejidoEmpresarial.ipynb`): valida, compila,
   hace commit, push y tag `vX.Y.Z` en
   [EnriqueForero/tejido_empresarial](https://github.com/EnriqueForero/tejido_empresarial).
2. En [railway.app](https://railway.app): **New Project → Deploy from GitHub
   repo** → elija el repositorio. Detecta `railway.toml` y construye el
   `Dockerfile` (Node 22 compila el frontend; Python 3.11 sirve la API).
3. **Variables** del servicio: `SF_ACCOUNT`, `SF_USER`, `SF_DATABASE`,
   `SF_SCHEMA`, `SF_WAREHOUSE`, `SF_ROLE`, `SF_PRIVATE_KEY_B64_1`
   (+ `SF_PRIVATE_KEY_PASSPHRASE_1` si aplica). Recomendado en un dominio
   público: `APP_BASIC_USER` y `APP_BASIC_PASSWORD`.
4. **Settings → Networking → Generate Domain**.
5. Verifique en este orden: `/api/health` → `/api/health?deep=true` →
   la portada → una búsqueda → una descarga.

| Síntoma en Railway | Causa probable | Solución |
|---|---|---|
| Build falla en `npm ci` | `frontend/package-lock.json` no viajó al repo | Publique de nuevo con el notebook (valida su presencia) |
| `/api/health` responde 503 | `APP_BASIC_USER` sin `APP_BASIC_PASSWORD` | Configure ambas o ninguna |
| `data_connection: missing_configuration` | Falta alguna variable `SF_*` | Complete las seis + la llave |
| `deep=true` responde 503 | Llave vencida o rol sin permisos | Rote la llave (`SF_PRIVATE_KEY_B64_2`) y revise el rol |
| La portada carga pero la búsqueda falla | Snowflake sin acceso a las tablas | Abra `/api/diagnostico`: señala el paso exacto. Guía completa en `DIAGNOSTICO_RAILWAY.md` |
